# 04 — Classical ML Classifiers

Train and 5-fold cross-validate Logistic Regression, SVM (RBF), and Random Forest on the CSP band-power features.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np

from models import get_sklearn_models
from train import cross_validate_sklearn
from utils import set_seed

set_seed(42)
DATA_DIR = Path('../data')

X_csp = np.load(DATA_DIR / 'X_csp.npy')
y = np.load(DATA_DIR / 'y.npy')
print('Features:', X_csp.shape)

## Cross-validate all classical models

In [ ]:
results = []
for name, model in get_sklearn_models().items():
    metrics = cross_validate_sklearn(X_csp, y, model)
    metrics['model'] = name
    results.append(metrics)
    print(f"{name:<22} acc={metrics['accuracy_mean']:.3f} "
          f"+/- {metrics['accuracy_std']:.3f}  f1={metrics['f1_macro_mean']:.3f}")

## Hyperparameter sweep (SVM example)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1]}
from sklearn.svm import SVC
grid = GridSearchCV(SVC(kernel='rbf', probability=True), param_grid, cv=5, scoring='accuracy')
grid.fit(X_csp, y)
print('Best params:', grid.best_params_)
print('Best CV accuracy:', grid.best_score_)

## Save intermediate results

In [ ]:
import json
Path('../results').mkdir(exist_ok=True)
with open('../results/classical_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved results/classical_metrics.json')